Türkiye Adresler .json -> https://github.com/metinyildirimnet/turkiye-adresler-json

# Model ( BiLSTM + Attention + CNN Modeli ) + Data Cleaning, Geographical Feature Extraction, Augmentation

In [ ]:
import os
import re
import sys
import json
import math
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

try:
    from rapidfuzz import process
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rapidfuzz"])
    from rapidfuzz import process

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Dropout,
    GlobalMaxPooling1D, MultiHeadAttention, LayerNormalization,
    BatchNormalization, Conv1D, Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# ---- 1) Dosya yolları ----
def find_paths():
    kaggle_train = "/kaggle/input/hepsiburada-hackathon-kaggle-etabi/train.csv"
    kaggle_test  = "/kaggle/input/hepsiburada-hackathon-kaggle-etabi/test.csv"

    colab_train = "/content/drive/MyDrive/hepsiburada/train.csv"
    colab_test  = "/content/drive/MyDrive/hepsiburada/test.csv"

    #JSON yolları
    colab_json_dir = "/content/drive/MyDrive/hepsiburada/turkiye-adresler-json-main/turkiye-adresler-json-main"

    local_train = "./train.csv"
    local_test  = "./test.csv"
    local_json_dir = "./turkiye-adresler-json-main"

    if os.path.exists(kaggle_train):
        print("Kaggle ortamı algılandı.")
        return kaggle_train, kaggle_test, "/kaggle/input/turkiye-adresler-json", "kaggle"
    elif os.path.exists(colab_train):
        print("Colab ortamı algılandı.")
        return colab_train, colab_test, colab_json_dir, "colab"
    elif os.path.exists(local_train):
        print("Lokal ortam algılandı.")
        return local_train, local_test, local_json_dir, "local"
    else:
        raise FileNotFoundError("Veri dosyaları bulunamadı.")

train_path, test_path, json_dir, env = find_paths()
print(f"Env: {env}, train: {train_path}, test: {test_path}, json_dir: {json_dir}")

# ---- 2) Veriler ----
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
print("Train shape:", train_df.shape, "Test shape:", test_df.shape)

# ---- 3) JSON verileri ----
def load_json_data(json_dir):
    data = {}
    with open(os.path.join(json_dir, "sehirler.json"), "r", encoding="utf-8") as f:
        data["cities"] = {item["sehir_adi"].lower(): item for item in json.load(f)}
    with open(os.path.join(json_dir, "ilceler.json"), "r", encoding="utf-8") as f:
        data["districts"] = {item["ilce_adi"].lower(): item for item in json.load(f)}

    all_neighborhoods = []
    for i in range(1, 5):
        try:
            with open(os.path.join(json_dir, f"mahalleler-{i}.json"), "r", encoding="utf-8") as f:
                all_neighborhoods.extend(json.load(f))
        except FileNotFoundError:
            pass
    data["neighborhoods"] = {item["mahalle_adi"].lower(): item for item in all_neighborhoods}

    try:
        with open(os.path.join(json_dir, "mahalleler.json"), "r", encoding="utf-8") as f:
            data["neighborhoods"].update({item["mahalle_adi"].lower(): item for item in json.load(f)})
    except FileNotFoundError:
        pass
    return data

geo_data = load_json_data(json_dir)
print(f"Yüklenen coğrafi veri: {len(geo_data['cities'])} şehir, {len(geo_data['districts'])} ilçe, {len(geo_data['neighborhoods'])} mahalle")

# --- 4) Adres Augmentation ---
_AUG_ABBREV_MAP = {
    "mahallesi": ["mah", "mh"],
    "caddesi": ["cad", "cd"],
    "sokak": ["sk", "sok"],
    "bulvari": ["blv", "bul"],
    "apartmani": ["apt", "aprt"],
    "numara": ["no", "nr"],
    "daire": ["da", "dr"],
}
def augment_address(text, p_aug=0.8):
    if random.random() > p_aug:
        return text
    words = text.split()
    augmented_words = []
    for word in words:
        for key, value_list in _AUG_ABBREV_MAP.items():
            if word.lower() in [key] + value_list:
                replacement_list = [key] + value_list
                replacement_list.remove(word.lower())
                if replacement_list:
                    word = random.choice(replacement_list)
                break
        augmented_words.append(word)
    text = " ".join(augmented_words)
    if random.random() < 0.3:
        text_list = list(text)
        op = random.choice(["delete", "insert", "swap"])
        if op == "delete" and len(text_list) > 5:
            text_list.pop(random.randint(0, len(text_list)-1))
        elif op == "insert":
            text_list.insert(random.randint(0, len(text_list)), random.choice("abcdei̇fghijklmnoprstu̇vzyxw"))
        elif op == "swap" and len(text_list) > 1:
            idx = random.randint(0, len(text_list)-2)
            text_list[idx], text_list[idx+1] = text_list[idx+1], text_list[idx]
        text = "".join(text_list)
    return text

train_df_aug = train_df.copy()
train_df_aug['address'] = train_df_aug['address'].progress_apply(lambda x: augment_address(x))
train_df = pd.concat([train_df, train_df_aug], ignore_index=True)
print("Augmentation sonrası train shape:", train_df.shape)

# ---- 5) Temizleme + Geo Feature extraction ----
_ABBREV_MAP = {"mahalle": ["mah","mh"], "caddesi":["cad","cd"], "sokak":["sk","sok"], "bulvari":["blv","bul"], "numara":["no","nr"], "daire":["dr","da"], "apartmani":["apt","aprt"]}
_VARIANTS = {k: v+[k] for k,v in _ABBREV_MAP.items()}

def _deasciify_tr(text):
    if not isinstance(text,str): text=str(text)
    tr_map=str.maketrans("İŞÇĞÜÖışçüğö","ISCGUOiscguo")
    text=text.translate(tr_map).lower().replace("i̇","i")
    return text

def _basic_regex_cleanup(text):
    return re.sub(r"\s+"," ", re.sub(r"[,:;./\\|()\[\]{}\-_]+"," ", text)).strip()

def _normalize_abbrevs_with_fuzzy(text, threshold=86):
    words=text.split(); out=[]
    for w in words:
        if w.isdigit(): out.append(w); continue
        match=None; best_score=-1; best_key=None
        for key,var_list in _VARIANTS.items():
            m=process.extractOne(w,var_list,score_cutoff=threshold)
            if m and m[1]>best_score: best_score=m[1]; best_key=key
        out.append(best_key if best_key else w)
    return " ".join(out)

def clean_and_extract_features(text, geo_data):
    if pd.isna(text): return "",0,0,0,""
    text=_deasciify_tr(text); text=_basic_regex_cleanup(text)
    found_city=found_district=found_neighborhood=0; found_geo_name=""
    words=text.split(); potential={}
    for i in range(len(words)-2):
        phrase=" ".join(words[i:i+3])
        if phrase in geo_data['neighborhoods']: potential['neighborhood']=(phrase,3)
    for i in range(len(words)-1):
        phrase=" ".join(words[i:i+2])
        if phrase in geo_data['neighborhoods']: potential['neighborhood']=(phrase,2)
        if phrase in geo_data['districts']: potential['district']=(phrase,2)
        if phrase in geo_data['cities']: potential['city']=(phrase,2)
    for w in words:
        if w in geo_data['neighborhoods']: potential['neighborhood']=(w,1)
        if w in geo_data['districts']: potential['district']=(w,1)
        if w in geo_data['cities']: potential['city']=(w,1)
    if 'neighborhood' in potential: found_neighborhood=1; found_geo_name=potential['neighborhood'][0]
    elif 'district' in potential: found_district=1; found_geo_name=potential['district'][0]
    elif 'city' in potential: found_city=1; found_geo_name=potential['city'][0]
    text=_normalize_abbrevs_with_fuzzy(_basic_regex_cleanup(text))
    return text,found_city,found_district,found_neighborhood,found_geo_name

train_df[["clean_address","found_city","found_district","found_neighborhood","found_geo_name"]] = train_df["address"].progress_apply(lambda x: pd.Series(clean_and_extract_features(x, geo_data)))
test_df[["clean_address","found_city","found_district","found_neighborhood","found_geo_name"]] = test_df["address"].progress_apply(lambda x: pd.Series(clean_and_extract_features(x, geo_data)))

# ---- 6) Label encoding ----
le = LabelEncoder()
y = le.fit_transform(train_df["label"].astype(str))
num_classes = len(le.classes_)
print("Unique labels:", num_classes)

# ---- 7) Tokenizer ----
max_words=60000; max_len=60
tokenizer=Tokenizer(num_words=max_words,oov_token="<unk>")
tokenizer.fit_on_texts(train_df["clean_address"].astype(str))
X_train_seq=tokenizer.texts_to_sequences(train_df["clean_address"].astype(str))
X_test_seq=tokenizer.texts_to_sequences(test_df["clean_address"].astype(str))
X_all_pad=pad_sequences(X_train_seq,maxlen=max_len,padding="post",truncating="post")
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding="post",truncating="post")
X_tr_geo_feat=train_df[["found_city","found_district","found_neighborhood"]].values
X_test_geo_feat=test_df[["found_city","found_district","found_neighborhood"]].values

# ---- 8) Train/Val split ----
X_tr,X_val,y_tr,y_val,X_tr_geo,X_val_geo=train_test_split(X_all_pad,y,X_tr_geo_feat,test_size=0.15,random_state=SEED,stratify=y)

# ---- 9) Focal loss ----
def make_focal_loss(gamma=2.5, alpha=0.5, num_classes=num_classes):
    def focal_loss(y_true,y_pred):
        y_true_oh=tf.one_hot(tf.cast(y_true,tf.int32),depth=num_classes)
        y_pred=tf.clip_by_value(y_pred,tf.keras.backend.epsilon(),1.-tf.keras.backend.epsilon())
        ce=-y_true_oh*tf.math.log(y_pred)
        weight=alpha*tf.pow(1.-y_pred,gamma)
        return tf.reduce_mean(tf.reduce_sum(weight*ce,axis=1))
    return focal_loss

# ---- 10) Model ----
inp_text = Input(shape=(max_len,), name="input_ids")
x_embed = Embedding(max_words, 128, input_length=max_len)(inp_text)
x_rnn = Bidirectional(LSTM(128, return_sequences=True))(x_embed)
x_att = MultiHeadAttention(num_heads=8, key_dim=64)(x_rnn, x_rnn)
x_att = LayerNormalization()(x_att)
x_att_gm = GlobalMaxPooling1D()(x_att)
x_cnn_3 = GlobalMaxPooling1D()(Conv1D(128, 3, activation='relu')(x_embed))
x_cnn_4 = GlobalMaxPooling1D()(Conv1D(128, 4, activation='relu')(x_embed))
x_cnn_5 = GlobalMaxPooling1D()(Conv1D(128, 5, activation='relu')(x_embed))
inp_geo = Input(shape=(3,))
x_geo = Dense(16, activation="relu")(inp_geo)
x_cat = Concatenate()([x_att_gm, x_cnn_3, x_cnn_4, x_cnn_5, x_geo])
x_cat = Dense(256, activation="relu")(x_cat)
x_cat = BatchNormalization()(x_cat)
x_cat = Dropout(0.5)(x_cat)
out = Dense(num_classes, activation="softmax")(x_cat)
model = Model(inputs=[inp_text, inp_geo], outputs=out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss=make_focal_loss(),
    metrics=["accuracy"]
)

model.summary()

# ---- 11) Callbacks ----
class F1Metrics(Callback):
    def __init__(self, X_val, X_val_geo, y_val, batch_size=256):
        super().__init__()
        self.X_val = X_val
        self.X_val_geo = X_val_geo
        self.y_val = y_val
        self.batch_size = batch_size

    def on_epoch_end(self, epoch, logs=None):
        preds = []
        for i in range(0, len(self.X_val), self.batch_size):
            p = self.model.predict(
                [self.X_val[i:i+self.batch_size], self.X_val_geo[i:i+self.batch_size]],
                batch_size=self.batch_size, verbose=0
            )
            preds.extend(np.argmax(p, axis=1))
        f1 = f1_score(self.y_val, np.array(preds), average="macro")
        print(f"\n — val_f1: {f1:.4f}")
        logs["val_f1"] = f1


class SubmissionCallback(Callback):
    def __init__(self, X_test_pad, X_test_geo_feat, test_ids, label_encoder, interval=10, out_dir="/content/drive/MyDrive/hepsiburada_checkpoints"):
        super().__init__()
        self.X_test_pad = X_test_pad
        self.X_test_geo_feat = X_test_geo_feat
        self.test_ids = test_ids
        self.le = label_encoder
        self.interval = interval
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)  # klasör yoksa oluştur

    def predict_in_batches(self, model, X_text, X_geo, batch_size=256):
        preds = []
        for i in range(0, len(X_text), batch_size):
            p = model.predict([X_text[i:i+batch_size], X_geo[i:i+batch_size]], batch_size=batch_size, verbose=0)
            preds.extend(np.argmax(p, axis=1))
        return np.array(preds)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.interval == 0:
            print(f"\n[Callback] Epoch {epoch+1}: submission.csv Drive'a kaydediliyor...")
            preds = self.predict_in_batches(self.model, self.X_test_pad, self.X_test_geo_feat)
            labels = self.le.inverse_transform(preds)
            submission_df = pd.DataFrame({"id": self.test_ids, "label": labels})
            out_path = os.path.join(self.out_dir, f"submission_epoch{epoch+1}.csv")
            submission_df.to_csv(out_path, index=False)
            print(f"[Callback] Submission kaydedildi: {out_path}")

early_stopping = EarlyStopping(
    monitor="val_loss", patience=8,
    restore_best_weights=True, verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.8, patience=6,
    min_lr=1e-4, verbose=1
)

f1_metrics = F1Metrics(X_val, X_val_geo, y_val)
submission_cb = SubmissionCallback(
    X_test_pad, X_test_geo_feat, test_df["id"].values, le, interval=10, out_dir="./"
)

# ---- 12) Eğitim ----
history = model.fit(
    [X_tr, X_tr_geo], y_tr,
    validation_data=([X_val, X_val_geo], y_val),
    epochs=120,
    batch_size=256,
    callbacks=[early_stopping, reduce_lr, f1_metrics, submission_cb],
    verbose=1
)


# ---- 13) Test Prediction ----
def predict_in_batches(model,X_text,X_geo,batch_size=256):
    preds=[]
    for i in range(0,len(X_text),batch_size):
        p=model.predict([X_text[i:i+batch_size],X_geo[i:i+batch_size]],batch_size=batch_size,verbose=0)
        preds.extend(np.argmax(p,axis=1))
    return np.array(preds)

y_val_pred=predict_in_batches(model,X_val,X_val_geo)
val_f1=f1_score(y_val,y_val_pred,average="macro")
print(f"\n📊 Final Validation Macro F1: {val_f1:.4f}")

test_preds_idx=predict_in_batches(model,X_test_pad,X_test_geo_feat)
test_preds_labels=le.inverse_transform(test_preds_idx)
submission_df=pd.DataFrame({"id":test_df["id"].values,"label":test_preds_labels})
out_path="./submission_no_ckpt.csv"
submission_df.to_csv(out_path,index=False)
print("🎉 Submission kaydedildi:", out_path)

# Önceden Kaydedilen .csv dosyaları ile Majority Voting (En yüksek alanlar sonuç baz alındı)

In [ ]:
import pandas as pd
from collections import Counter

files = [
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch120.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch_60.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch_70.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch_80.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_no_ckpt.csv"
]

dfs = [pd.read_csv(f) for f in files]

id_col = dfs[0].columns[0]
pred_col = dfs[0].columns[1] if len(dfs[0].columns) > 1 else dfs[0].columns[0]

n_rows = len(dfs[0])
changed_count = 0
majority_log = []

final_preds = []

for i in range(n_rows):
    row_preds = [df.iloc[i][pred_col] for df in dfs]
    most_common, freq = Counter(row_preds).most_common(1)[0]

    if len(set(row_preds)) > 1:
        changed_count += 1
        majority_log.append((i, row_preds, most_common))

    final_preds.append(most_common)

if len(dfs[0].columns) > 1:
    final_df = pd.DataFrame({
        id_col: dfs[0][id_col],
        pred_col: final_preds
    })
else:
    final_df = pd.DataFrame({pred_col: final_preds})

output_path = "/content/drive/MyDrive/hepsiburada/predictions/submission_majority_votedd.csv"
final_df.to_csv(output_path, index=False)


print(f"Toplam değişen satır sayısı: {changed_count}")
print("\nMajority Voting sonuçları (ilk 20 farklı satır):")
for idx, row_preds, vote in majority_log[:20]:
    print(f"Satır {idx}: {row_preds} -> seçilen: {vote}")


In [ ]:
import pandas as pd
from collections import Counter

files = [
    "/content/drive/MyDrive/hepsiburada/predictions/submission_bilstm_focal_safe.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch120.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch_60.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch_70.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_epoch_80.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_majority_voted.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_majority_voted2.0.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_majority_votedd.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_no_ckpt.csv",
    "/content/drive/MyDrive/hepsiburada/predictions/submission_v2_improved_model.csv"
]

priority_order = [
    "submission_majority_voted2.0",
    "submission_majority_votedd",
    "epoch_60",
    "epoch_80",
    "epoch_70",
    "submission_bilstm_focal_safe"
]

dfs = [pd.read_csv(f) for f in files]

id_col = dfs[0].columns[0]
pred_col = dfs[0].columns[1] if len(dfs[0].columns) > 1 else dfs[0].columns[0]

n_rows = len(dfs[0])
changed_count = 0
majority_log = []

final_preds = []

def resolve_tie(row_preds, row_index):
    """Eşitlik durumunda öncelik sırasına göre seçim yap"""
    for priority in priority_order:
        for f, df in zip(files, dfs):
            if priority in f:
                candidate = df.iloc[row_index][pred_col]
                if row_preds.count(candidate) > 0:
                    return candidate
    return row_preds[0]

for i in range(n_rows):
    row_preds = [df.iloc[i][pred_col] for df in dfs]
    counter = Counter(row_preds)
    most_common, freq = counter.most_common(1)[0]

    if len(set(row_preds)) > 1:
        max_freq = freq
        tied = [val for val, c in counter.items() if c == max_freq]

        if len(tied) > 1:
            chosen = resolve_tie(row_preds, i)
        else:
            chosen = most_common

        changed_count += 1
        majority_log.append((i, row_preds, chosen))
    else:
        chosen = most_common

    final_preds.append(chosen)

if len(dfs[0].columns) > 1:
    final_df = pd.DataFrame({
        id_col: dfs[0][id_col],
        pred_col: final_preds
    })
else:
    final_df = pd.DataFrame({pred_col: final_preds})

output_path = "/content/drive/MyDrive/hepsiburada/predictions/submission_majority_final.csv"
final_df.to_csv(output_path, index=False)

print(f"Toplam değişen satır sayısı: {changed_count}")
print("\nMajority Voting sonuçları (ilk 20 farklı satır):")
for idx, row_preds, vote in majority_log[:20]:
    print(f"Satır {idx}: {row_preds} -> seçilen: {vote}")
